# 🐒 Phân nhóm và Tiền xử lý Dữ liệu BioListen VN (Bird, Frog, Insect)

Notebook này được thiết kế để chuẩn bị tập dữ liệu gộp nhóm phục vụ cho nhánh dự đoán động vật `species_head` với 3 nhóm chính hiện tại:
- **Bird (Chim)**
- **Frog (Ếch)**
- **Insect (Côn trùng)**

### 📂 Cơ cấu thư mục đầu ra trên Google Drive:
Mô hình sẽ lưu trữ trực tiếp các tệp `.pt` (PyTorch tensor) đã tiền xử lý vào cấu trúc thư mục:
- `BioListenVN/grouping/Bird/`
- `BioListenVN/grouping/Frog/`
- `BioListenVN/grouping/Insect/`

### 🛠️ Nguyên tắc hoạt động:
1. **Không giải nén xuống đĩa Colab:** Đọc và xử lý hoàn toàn trong RAM thông qua `zipfile` và ghi trực tiếp lên Google Drive để tránh tràn ổ cứng.
2. **Phân nhóm RFCx:** Gộp 24 species ID thành Chim và Ếch dựa trên kết quả kiểm tra nhãn thủ công (Ếch: `0, 2, 4, 12, 13, 15, 18, 19, 20` | Chim: các ID còn lại).
3. **Tiền xử lý Insect (Zenodo):** Đọc các file WAV thô của côn trùng từ `insect_Train.zip`/`insect_Validation.zip`, chuyển đổi sang định dạng tensor Mel-spectrogram `(3, 224, 224)` đồng bộ với các loài khác.

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q torchaudio soundfile pandas tqdm

from google.colab import drive
import os

# Kết nối Google Drive
if not os.path.exists('/content/drive'):
    print("Đang kết nối đến Google Drive...")
    drive.mount('/content/drive')
else:
    print("Google Drive đã kết nối [OK]")

In [ ]:
import zipfile
import shutil
import io
import pandas as pd
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
import soundfile as sf
from tqdm import tqdm

DRIVE_DIR = '/content/drive/MyDrive/Datasets/BioListenVN'
GROUPING_DIR = os.path.join(DRIVE_DIR, 'grouping')

# Tạo các thư mục lưu trữ
os.makedirs(os.path.join(GROUPING_DIR, 'Bird'), exist_ok=True)
os.makedirs(os.path.join(GROUPING_DIR, 'Frog'), exist_ok=True)
os.makedirs(os.path.join(GROUPING_DIR, 'Insect'), exist_ok=True)

print("[OK] Đã tạo/kiểm tra cấu trúc thư mục gộp nhóm trên Drive:")
print(f" - {os.path.join(GROUPING_DIR, 'Bird')}")
print(f" - {os.path.join(GROUPING_DIR, 'Frog')}")
print(f" - {os.path.join(GROUPING_DIR, 'Insect')}")

## 1. Phân nhóm Dữ liệu RFCx (Chim & Ếch)

Dựa trên kết quả manual check:
- **Ếch (Frog) species_id:** `[0, 2, 4, 12, 13, 15, 18, 19, 20]`
- **Chim (Bird) species_id:** Các species_id còn lại.

In [ ]:
FROG_SPECIES = [0, 2, 4, 12, 13, 15, 18, 19, 20]

rfcx_csv_path = os.path.join(DRIVE_DIR, 'processed', 'rfcx_tp_processed_metadata.csv')
rfcx_zip_path = os.path.join(DRIVE_DIR, 'processed', 'rfcx_tp_processed.zip')

if os.path.exists(rfcx_csv_path) and os.path.exists(rfcx_zip_path):
    df_rfcx = pd.read_csv(rfcx_csv_path)
    print(f"Đọc thành công metadata RFCx. Số lượng mẫu: {len(df_rfcx)}")
    
    frog_count = 0
    bird_count = 0
    
    print("Đang tiến hành giải nén và phân nhóm vào các thư mục tương ứng...")
    with zipfile.ZipFile(rfcx_zip_path, 'r') as z:
        all_members = z.namelist()
        
        for _, row in tqdm(df_rfcx.iterrows(), total=len(df_rfcx)):
            pt_name = row['processed_pt_filename']
            species_id = int(row['species_id'])
            
            # Xác định thư mục đích
            group_name = 'Frog' if species_id in FROG_SPECIES else 'Bird'
            dest_path = os.path.join(GROUPING_DIR, group_name, pt_name)
            
            # Tìm file trong ZIP và trích xuất
            member = [m for m in all_members if m.endswith(pt_name)]
            if member:
                with z.open(member[0]) as src:
                    with open(dest_path, 'wb') as dst:
                        dst.write(src.read())
                if group_name == 'Frog':
                    frog_count += 1
                else:
                    bird_count += 1
                    
    print(f"[Xong] Đã phân nhóm thành công {frog_count} file Ếch và {bird_count} file Chim.")
else:
    print("❌ Lỗi: Không tìm thấy file metadata rfcx_tp_processed_metadata.csv hoặc rfcx_tp_processed.zip trên Drive.")

## 2. Tiền xử lý & Tạo Dữ liệu Côn trùng (InsectSet459)

Ở đây, chúng ta định nghĩa class tiền xử lý âm thanh sử dụng cấu hình Mel-spectrogram đồng bộ với RFCx và ESC-50 (`sample_rate=32000`, `fmin=200`, `fmax=15000` để lọc gió và tạp âm nền).

In [ ]:
class InsectAudioPreprocessor:
    def __init__(self):
        self.sample_rate = 32000
        self.duration_sec = 5
        self.target_samples = self.sample_rate * self.duration_sec
        self.n_fft = 2048
        self.hop_length = 512
        self.n_mels = 128
        self.fmin = 200
        self.fmax = 15000
        
    def preprocess_waveform(self, y, sr):
        # Chuyển mono
        if len(y.shape) > 1:
            y = np.mean(y, axis=1)
            
        # Chuyển tensor
        waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
        
        # Resample nếu lệch tần số lấy mẫu
        if sr != self.sample_rate:
            resampler = T.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)
            
        # Cắt/Pad về đúng 5 giây (160,000 samples)
        total_samples = waveform.shape[1]
        if total_samples > self.target_samples:
            # Cắt 5 giây đầu tiên
            waveform = waveform[:, :self.target_samples]
        elif total_samples < self.target_samples:
            pad_len = self.target_samples - total_samples
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            
        # Trích xuất Mel-spectrogram
        mel_transform = T.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            n_mels=self.n_mels,
            f_min=self.fmin,
            f_max=self.fmax
        )
        mel_spec = mel_transform(waveform)
        
        # Đổi sang dB scale
        amplitude_to_db = T.AmplitudeToDB()
        spec_db = amplitude_to_db(mel_spec)
        
        # Chuẩn hóa về [0, 1]
        min_val = spec_db.min()
        max_val = spec_db.max()
        if max_val - min_val > 1e-9:
            spec_norm = (spec_db - min_val) / (max_val - min_val)
        else:
            spec_norm = torch.zeros_like(spec_db)
            
        # Nội suy resize về (224, 224)
        spec_resized = torch.nn.functional.interpolate(
            spec_norm.unsqueeze(0),
            size=(224, 224),
            mode='bilinear',
            align_corners=False
        ).squeeze(0)
        
        # Lặp thành 3 kênh RGB giống nhau
        mel_rgb = spec_resized.repeat(3, 1, 1)
        return mel_rgb

In [ ]:
zenodo_csv = os.path.join(DRIVE_DIR, 'raw_zips', 'InsectSet459_Train_Val_Test_Annotation.csv')
zip_train = os.path.join(DRIVE_DIR, 'raw_zips', 'insect_Train.zip')
zip_val = os.path.join(DRIVE_DIR, 'raw_zips', 'insect_Validation.zip')

if os.path.exists(zenodo_csv):
    df_insect = pd.read_csv(zenodo_csv)
    print(f"Đọc thành công metadata Côn trùng. Số lượng mẫu: {len(df_insect)}")
    
    # Lập chỉ mục file WAV trong các file ZIP
    file_to_zip = {}
    def index_zip(path):
        if os.path.exists(path):
            print(f"Đang lập chỉ mục {os.path.basename(path)}...")
            with zipfile.ZipFile(path, 'r') as z:
                for name in z.namelist():
                    if name.endswith('.wav'):
                        file_to_zip[os.path.basename(name)] = (path, name)
                        
    index_zip(zip_train)
    index_zip(zip_val)
    print(f"Lập chỉ mục xong. Tổng số file WAV tìm thấy: {len(file_to_zip)}")
    
    # Lấy ngẫu nhiên khoảng 1,500 mẫu côn trùng để cân bằng lớp dữ liệu với Chim và Ếch
    insect_samples = df_insect[df_insect['file_name'].isin(file_to_zip.keys())]
    sample_size = min(1500, len(insect_samples))
    selected_df = insect_samples.sample(sample_size, random_state=42)
    print(f"Đã chọn {len(selected_df)} mẫu côn trùng ngẫu nhiên để tiến hành tiền xử lý...")
    
    preprocessor = InsectAudioPreprocessor()
    insect_count = 0
    
    # Cache handles
    z_handles = {}
    
    try:
        for _, row in tqdm(selected_df.iterrows(), total=len(selected_df)):
            fname = row['file_name']
            zip_p, member_p = file_to_zip[fname]
            
            if zip_p not in z_handles:
                z_handles[zip_p] = zipfile.ZipFile(zip_p, 'r')
            z = z_handles[zip_p]
            
            try:
                # Đọc nhị phân và tiền xử lý
                with z.open(member_p) as f:
                    audio_data = io.BytesIO(f.read())
                    y, sr = sf.read(audio_data)
                    
                    # Trích xuất tensor RGB
                    mel_rgb = preprocessor.preprocess_waveform(y, sr)
                    
                    # Lưu tensor trực tiếp lên Drive
                    pt_name = fname.replace('.mp3', '.pt').replace('.wav', '.pt')
                    dest_path = os.path.join(GROUPING_DIR, 'Insect', pt_name)
                    torch.save(mel_rgb, dest_path)
                    insect_count += 1
            except Exception as e:
                print(f"Lỗi xử lý file {fname}: {e}")
    finally:
        for zh in z_handles.values():
            zh.close()
            
    print(f"[Xong] Đã hoàn tất tiền xử lý và lưu {insect_count} file Côn trùng vào grouping/Insect/.")
else:
    print("❌ Lỗi: Không tìm thấy file metadata InsectSet459_Train_Val_Test_Annotation.csv trên Drive.")